# Comment déployer des modèles de Machine Learning comme un pro

Notebook de l'article [Comment déployer des modèles de Machine Learning comme un pro](https://moncoachdata.com/blog/comment-deployer-des-modeles-de-machine-learning-comme-un-pro/) de MonCoachData.

Le notebook entraîne un modèle, le sauvegarde, démarre une API Flask puis gunicorn, la teste avec `curl`, et détecte une dérive des données.

## Préparation

On installe les paquets qui ne sont pas déjà présents sur Google Colab.

In [1]:
!pip install -q gunicorn fastapi httpx

## Versions utilisées

In [2]:
import json
import subprocess
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
import sklearn

print("Python", sys.version.split()[0])
print("scikit-learn", sklearn.__version__)
print("joblib", joblib.__version__)

def sh(commande):
    """Lance une commande dans le terminal et affiche sa sortie."""
    resultat = subprocess.run(commande, shell=True, capture_output=True, text=True)
    sortie = (resultat.stdout + resultat.stderr).strip()
    if sortie:
        print(sortie)

Python 3.13.18
scikit-learn 1.6.1
joblib 1.6.0


## Étape 1 : entraîner le modèle

Le fichier `data.csv` est créé à partir du jeu de données Iris, livré avec scikit-learn.

In [3]:
from sklearn.datasets import load_iris

# Le jeu de données Iris (150 fleurs, 4 mesures, 3 espèces) est livré avec scikit-learn
load_iris(as_frame=True).frame.to_csv("data.csv", index=False)

df = pd.read_csv("data.csv")
print(df.shape)
print(df["target"].value_counts().sort_index().to_dict())
df.head(3)

(150, 5)
{0: 50, 1: 50, 2: 50}


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Charger un échantillon de données
df = pd.read_csv("data.csv")

# Séparer en features et target
X = df.drop("target", axis=1)
y = df["target"]

# Diviser en train / test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entraîner un modèle de régression logistique
model = LogisticRegression()
model.fit(X_train, y_train)

# Évaluer le modèle
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

Accuracy: 1.0


## Étape 2 : sauvegarder le modèle

In [5]:
# Sauvegarder le modèle dans un fichier
joblib.dump(model, "logistic_regression_model.pkl")

taille = Path("logistic_regression_model.pkl").stat().st_size
print(f"Fichier : logistic_regression_model.pkl ({taille} octets)")

# Vérification : le modèle rechargé donne exactement les mêmes prédictions
modele_recharge = joblib.load("logistic_regression_model.pkl")
print("Mêmes prédictions après rechargement :", (modele_recharge.predict(X_test) == y_pred).all())

Fichier : logistic_regression_model.pkl (1327 octets)
Mêmes prédictions après rechargement : True


### Piège 1 : la version de scikit-learn

On fait comme si le modèle avait été sauvegardé avec scikit-learn 1.0.2.

In [ ]:
import sklearn.base

vraie_version = sklearn.base.__version__
sklearn.base.__version__ = "1.0.2"          # on fait comme si le modèle avait été sauvegardé avec une ancienne version
joblib.dump(model, "ancien_modele.pkl")
sklearn.base.__version__ = vraie_version

with warnings.catch_warnings(record=True) as avertissements:
    warnings.simplefilter("always")
    joblib.load("ancien_modele.pkl")

print(avertissements[0].category.__name__)
print(str(avertissements[0].message).split(" For more info")[0])

### Piège 2 : la sécurité

Charger un fichier pickle ou joblib peut exécuter du code : ne charge que des fichiers de confiance.

In [7]:
class Piege:
    def __reduce__(self):
        # Dit à pickle : "pour reconstruire cet objet, appelle print(...)"
        return (print, ("Du code s'est exécuté pendant le chargement du fichier !",))

joblib.dump(Piege(), "inoffensif.pkl")
joblib.load("inoffensif.pkl")

Du code s'est exécuté pendant le chargement du fichier !


## Étape 3 : le traitement par lots

Le modèle tourne sur un fichier, sans API.

In [8]:
# Traitement par lots : le modèle tourne sur un fichier, sans API
nouveaux = df.drop(columns="target").sample(8, random_state=1)
nouveaux.to_csv("nouveaux.csv", index=False)

modele = joblib.load("logistic_regression_model.pkl")
lot = pd.read_csv("nouveaux.csv")
lot["prediction"] = modele.predict(lot)
lot.to_csv("predictions.csv", index=False)

print(lot.to_string(index=False))

 sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  prediction
               5.8               4.0                1.2               0.2           0
               5.1               2.5                3.0               1.1           1
               6.6               3.0                4.4               1.4           1
               5.4               3.9                1.3               0.4           0
               7.9               3.8                6.4               2.0           2
               6.3               3.3                4.7               1.6           1
               6.9               3.1                5.1               2.3           2
               5.1               3.8                1.9               0.4           0


## Étape 4 : l'API Flask

On écrit `app.py` (la cellule `%%writefile` crée le fichier).

In [9]:
%%writefile app.py
from flask import Flask, request, jsonify
import joblib
import pandas as pd

# Charger le modèle entraîné
model = joblib.load("logistic_regression_model.pkl")
ESPECES = ["setosa", "versicolor", "virginica"]

app = Flask(__name__)


@app.route("/health")
def health():
    return jsonify({"status": "ok"})


@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True)   # None si le JSON est absent ou invalide
    features = data.get("features") if isinstance(data, dict) else None
    if not isinstance(features, list) or len(features) != 4 \
            or not all(isinstance(x, (int, float)) for x in features):
        return jsonify({"erreur": "envoie 4 nombres dans 'features'"}), 400
    # Même noms de colonnes qu'à l'entraînement, sinon scikit-learn affiche un avertissement
    X = pd.DataFrame([features], columns=model.feature_names_in_)
    classe = int(model.predict(X)[0])
    return jsonify({"prediction": classe, "espece": ESPECES[classe]})


if __name__ == "__main__":
    app.run(port=5000)   # surtout pas debug=True en production

Writing app.py


### Lancer l'API

On démarre l'API en arrière-plan et on attend qu'elle réponde.

In [10]:
log = open("flask.log", "w")
serveur = subprocess.Popen([sys.executable, "app.py"], stdout=log, stderr=subprocess.STDOUT)

# On attend que l'API réponde
for _ in range(100):
    try:
        print(requests.get("http://127.0.0.1:5000/health", timeout=1).json())
        break
    except requests.RequestException:   # pas encore prêt (connexion refusée ou réponse trop lente)
        time.sleep(0.2)

{'status': 'ok'}


### Appeler l'API avec `curl`

In [11]:
sh("""curl -s -X POST http://localhost:5000/predict -H "Content-Type: application/json" -d '{"features": [5.1, 3.5, 1.4, 0.2]}'""")

{"espece":"setosa","prediction":0}


### Erreur et latence

Une requête invalide reçoit une erreur 400. La latence dépend de la machine.

In [12]:
# Une requête invalide reçoit une erreur claire, pas un plantage
reponse = requests.post("http://127.0.0.1:5000/predict", json={"features": [5.1, 3.5]})
print(reponse.status_code, reponse.json())

# Latence moyenne sur 200 requêtes (serveur de développement, sur ma machine)
session = requests.Session()
debut = time.perf_counter()
for _ in range(200):
    session.post("http://127.0.0.1:5000/predict", json={"features": [5.1, 3.5, 1.4, 0.2]})
print(f"Latence moyenne : {(time.perf_counter() - debut) / 200 * 1000:.1f} ms par requête")

400 {'erreur': "envoie 4 nombres dans 'features'"}
Latence moyenne : 4.9 ms par requête


### Arrêter l'API

In [13]:
serveur.terminate()
serveur.wait()
log.close()
print("Serveur arrêté.")
print("Requêtes POST /predict dans le journal :", Path("flask.log").read_text().count("POST /predict"))

Serveur arrêté.
Requêtes POST /predict dans le journal : 202


### gunicorn : le serveur de production

On lance l'API avec deux workers et on refait le test.

In [14]:
log = open("gunicorn.log", "w")
prod = subprocess.Popen([sys.executable, "-m", "gunicorn", "-w", "2", "-b", "127.0.0.1:8000", "app:app"],
                        stdout=log, stderr=subprocess.STDOUT)
for _ in range(100):
    try:
        requests.get("http://127.0.0.1:8000/health", timeout=1)
        break
    except requests.RequestException:   # pas encore prêt (connexion refusée ou réponse trop lente)
        time.sleep(0.2)

print(requests.post("http://127.0.0.1:8000/predict", json={"features": [6.7, 3.0, 5.2, 2.3]}).json())

session = requests.Session()
debut = time.perf_counter()
for _ in range(200):
    session.post("http://127.0.0.1:8000/predict", json={"features": [6.7, 3.0, 5.2, 2.3]})
print(f"Latence moyenne : {(time.perf_counter() - debut) / 200 * 1000:.1f} ms par requête")

prod.terminate()
prod.wait()
log.close()
print("Workers gunicorn démarrés :", Path("gunicorn.log").read_text().count("Booting worker"))

{'espece': 'virginica', 'prediction': 2}
Latence moyenne : 3.0 ms par requête
Workers gunicorn démarrés : 2


### La même API avec FastAPI

In [15]:
%%writefile app_fastapi.py
from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib
import pandas as pd

model = joblib.load("logistic_regression_model.pkl")
app = FastAPI()


class Mesures(BaseModel):
    features: list[float] = Field(min_length=4, max_length=4)


@app.post("/predict")
def predict(mesures: Mesures):
    X = pd.DataFrame([mesures.features], columns=model.feature_names_in_)
    return {"prediction": int(model.predict(X)[0])}

Writing app_fastapi.py


On teste FastAPI sans serveur, avec `TestClient`.

In [16]:
import importlib
from fastapi.testclient import TestClient

importlib.invalidate_caches()
import app_fastapi

client = TestClient(app_fastapi.app)
valide = client.post("/predict", json={"features": [5.1, 3.5, 1.4, 0.2]})
print(valide.status_code, valide.json())

invalide = client.post("/predict", json={"features": [5.1, 3.5]})
print(invalide.status_code, invalide.json()["detail"][0]["msg"])

print("Documentation automatique (/docs) :", client.get("/docs").status_code)

200 {'prediction': 0}
422 List should have at least 4 items after validation, not 2
Documentation automatique (/docs) : 200


## Étape 5 : Docker

On écrit le `Dockerfile`. Docker n'est pas disponible dans Colab : pour construire l'image, lance `docker build -t ml-model-api .` puis `docker run -p 5000:5000 ml-model-api` sur ton ordinateur.

In [17]:
%%writefile Dockerfile
# Image Python officielle, version légère
FROM python:3.13-slim

WORKDIR /app

# On copie d'abord requirements.txt : Docker réutilise le cache tant que les dépendances ne changent pas
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Puis le code et le modèle
COPY app.py logistic_regression_model.pkl ./

# Le port 5000 est celui de l'API
EXPOSE 5000

# gunicorn plutôt que le serveur de développement de Flask
CMD ["gunicorn", "-w", "2", "-b", "0.0.0.0:5000", "app:app"]

Writing Dockerfile


On génère `requirements.txt` avec les versions exactes installées.

In [18]:
dependances = ["flask", "gunicorn", "joblib", "pandas", "scikit-learn"]
from importlib.metadata import version
Path("requirements.txt").write_text("\n".join(f"{nom}=={version(nom)}" for nom in dependances) + "\n")
Path(".dockerignore").write_text("__pycache__/\n*.log\ndata.csv\n")

print(sorted(p.name for p in Path(".").iterdir() if p.name in ("Dockerfile", "requirements.txt", ".dockerignore", "app.py")))
print(Path("requirements.txt").read_text())

['.dockerignore', 'Dockerfile', 'app.py', 'requirements.txt']
flask==3.1.3
gunicorn==27.0.0
joblib==1.6.0
pandas==2.2.3
scikit-learn==1.6.1



## Étape 7 : surveiller le modèle

Simulation d'un capteur déréglé et détection de la dérive des données.

In [19]:
from scipy.stats import ks_2samp

# Scénario : un capteur se dérègle et mesure les pétales 40 % trop petits
prod_normale = X_test.copy()
prod_derivee = X_test.copy()
for colonne in ["petal length (cm)", "petal width (cm)"]:
    prod_derivee[colonne] = prod_derivee[colonne] * 0.6

print(f"Accuracy, données normales : {accuracy_score(y_test, model.predict(prod_normale)):.2f}")
print(f"Accuracy, capteur déréglé  : {accuracy_score(y_test, model.predict(prod_derivee)):.2f}")

# Test de Kolmogorov-Smirnov : la distribution a-t-elle changé par rapport à l'entraînement ?
print()
print(f"{'variable':<20}{'p normal':>10}{'p dérivé':>10}")
for colonne in X_train.columns:
    p_normal = ks_2samp(X_train[colonne], prod_normale[colonne]).pvalue
    p_derive = ks_2samp(X_train[colonne], prod_derivee[colonne]).pvalue
    alerte = "  ALERTE" if p_derive < 0.01 else ""
    print(f"{colonne:<20}{p_normal:>10.3f}{p_derive:>10.4f}{alerte}")

Accuracy, données normales : 1.00
Accuracy, capteur déréglé  : 0.60

variable              p normal  p dérivé
sepal length (cm)        0.880    0.8799
sepal width (cm)         1.000    0.9998
petal length (cm)        0.880    0.0000  ALERTE
petal width (cm)         0.760    0.0000  ALERTE


### Versioning

Une fiche enregistrée à côté du modèle.

In [20]:
import hashlib

Path("modeles").mkdir(exist_ok=True)
fichier = Path("modeles/iris-v1.joblib")
joblib.dump(model, fichier)

fiche = {
    "version": "v1",
    "sklearn": sklearn.__version__,
    "accuracy_test": float(accuracy_score(y_test, y_pred)),
    "empreinte_sha256": hashlib.sha256(fichier.read_bytes()).hexdigest()[:16],
}
Path("modeles/iris-v1.json").write_text(json.dumps(fiche, indent=2))
print(Path("modeles/iris-v1.json").read_text())

{
  "version": "v1",
  "sklearn": "1.6.1",
  "accuracy_test": 1.0,
  "empreinte_sha256": "15456af30ea306d7"
}
